# WGAN-GP Training — IASD Dauphine

Ce notebook entraîne le GAN sur Colab (GPU T4 gratuit) et pousse les checkpoints vers GitHub.

**Étapes :**
1. Vérifier le GPU
2. Cloner le repo + installer les dépendances
3. Entraîner (~25-35 min sur T4)
4. Vérifier les checkpoints
5. Pousser vers GitHub

> **Avant de lancer :** activer le GPU dans *Runtime → Change runtime type → T4 GPU*

## Cellule 1 — GPU check + clone du repo

In [1]:
import torch, os, subprocess

if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("ATTENTION : pas de GPU détecté. Va dans Runtime → Change runtime type → T4 GPU.")


import os
TOKEN = "YOUR_GITHUB_TOKEN"  # colle ton token ici
os.environ["GH_TOKEN"] = TOKEN

!git clone https://{TOKEN}@github.com/Master-IASD/iasdapp-project-frangigan.git
%cd iasdapp-project-frangigan
!pip install -r requirements.txt


GPU : Tesla T4
VRAM : 15.6 GB
Cloning into 'iasdapp-project-frangigan'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 51 (delta 10), reused 9 (delta 4), pack-reused 26 (from 3)
Receiving objects: 100% (51/51), 15.74 MiB | 16.15 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/iasdapp-project-frangigan


## Cellule 2 — Installation des dépendances

In [2]:
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print("Dépendances installées.")

Dépendances installées.


In [3]:
!nvidia-smi

Wed Mar 11 11:49:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cellule 3 — Entraînement WGAN-GP (300 epochs, ~25-35 min sur T4)

In [4]:
!python train.py --mode wgan --epochs 500 --batch_size 128 --n_critic 5 --lambda_gp 10 --ema_decay 0.9999

Using device: CUDA
Using 1 GPUs.
Dataset loading...
100% 9.91M/9.91M [00:02<00:00, 4.82MB/s]
100% 28.9k/28.9k [00:00<00:00, 130kB/s]
100% 1.65M/1.65M [00:01<00:00, 1.24MB/s]
100% 4.54k/4.54k [00:00<00:00, 15.5MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Dataset loaded.
Model loading...
Model loaded.
Start training (mode=wgan, epochs=300, batch=128):
Epoch 1/300:   0% 0/468 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max numb

## Cellule 4 — Vérification des checkpoints

In [8]:
import torch
from model import Generator, WGANCritic
from utils import load_model, load_critic

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

G = Generator(784).to(device)
G = load_model(G, 'checkpoints', device)
G.eval()

D = WGANCritic(784).to(device)
D = load_critic(D, 'checkpoints', device)
D.eval()

# Test forward
z   = torch.randn(4, 100, device=device)
x   = G(z)
out = D(x)
print(f"G output shape : {x.shape}")
print(f"D output shape : {out.shape}")

# Confirmer que D.pth est au format SN
d_keys = list(torch.load('checkpoints/D.pth', map_location='cpu').keys())
is_sn  = any('weight_orig' in k for k in d_keys)
print(f"Format D.pth : {'Spectral Norm ✅' if is_sn else 'Plain ⚠️'}")
print("Clés (extrait) :", d_keys[:6])

G output shape : torch.Size([4, 784])
D output shape : torch.Size([4, 1])
Format D.pth : Spectral Norm ✅
Clés (extrait) : ['fc1.bias', 'fc1.weight_orig', 'fc1.weight_u', 'fc1.weight_v', 'fc2.bias', 'fc2.weight_orig']


## Cellule 5A — Push vers GitHub (depuis Colab)

Créer un token GitHub : **Settings → Developer Settings → Personal Access Tokens → Tokens (classic)**  
Scope nécessaire : `repo`

Ou ajouter le token dans **Colab Secrets** (icône 🔑 dans la barre latérale) avec la clé `GITHUB_TOKEN`.

In [9]:
# Option A : push depuis Colab
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"   # ← coller ton token ici si tu ne veux pas utiliser les Secrets

try:
    from google.colab import userdata
    token = GITHUB_TOKEN or userdata.get('GITHUB_TOKEN')
except Exception:
    token = GITHUB_TOKEN

if not token:
    print("Token manquant — utilise la Cellule 5B pour télécharger les checkpoints.")
else:
    subprocess.run(['git', 'config', 'user.email', 'colab@train'], check=True)
    subprocess.run(['git', 'config', 'user.name',  'Colab Training'], check=True)
    subprocess.run(['git', 'add', 'checkpoints/G.pth', 'checkpoints/D.pth'], check=True)
    subprocess.run(['git', 'commit', '-m',
        'Add retrained checkpoints (WGAN-GP SN+LN, 300 epochs, T4)'], check=True)
    remote = f"https://{token}@github.com/Master-IASD/iasdapp-project-frangigan.git"
    subprocess.run(['git', 'push', remote, 'main'], check=True)
    print("✅ Push OK — checkpoints mis à jour sur GitHub.")

✅ Push OK — checkpoints mis à jour sur GitHub.


## Cellule 5B — Téléchargement local (alternative si pas de token)

In [7]:
# Option B : télécharger les fichiers localement
from google.colab import files
files.download('checkpoints/G.pth')
files.download('checkpoints/D.pth')
print("Téléchargement lancé. Ensuite, depuis ton terminal local :")
print("  cp ~/Downloads/G.pth checkpoints/")
print("  cp ~/Downloads/D.pth checkpoints/")
print("  git add checkpoints/ && git commit -m 'Retrained checkpoints' && git push")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Téléchargement lancé. Ensuite, depuis ton terminal local :
  cp ~/Downloads/G.pth checkpoints/
  cp ~/Downloads/D.pth checkpoints/
  git add checkpoints/ && git commit -m 'Retrained checkpoints' && git push


In [ ]:

# ============================================================
# Figure 5b — Negative critic loss : train vs validation
# 1000 images MNIST, WGAN-GP from scratch (n'affecte pas les checkpoints)
# ============================================================
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from model import Generator, WGANCritic

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Données : 1000 images train, 1000 images val ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
mnist_train = datasets.MNIST(root='data', train=True,  transform=transform, download=True)
mnist_test  = datasets.MNIST(root='data', train=False, transform=transform, download=True)

torch.manual_seed(42)
train_subset = torch.utils.data.Subset(mnist_train, torch.randperm(len(mnist_train))[:1000])
val_subset   = torch.utils.data.Subset(mnist_test,  torch.randperm(len(mnist_test))[:1000])

train_loader = torch.utils.data.DataLoader(train_subset, batch_size=64, shuffle=True,  drop_last=True)
val_loader   = torch.utils.data.DataLoader(val_subset,   batch_size=64, shuffle=False)

# --- Modèles réinitialisés (séparés des checkpoints) ---
G5 = Generator(784).to(device)
D5 = WGANCritic(784).to(device)
G_opt = optim.Adam(G5.parameters(), lr=1e-4, betas=(0.0, 0.9))
D_opt = optim.Adam(D5.parameters(), lr=1e-4, betas=(0.0, 0.9))

LAMBDA_GP = 10
N_CRITIC  = 5
EPOCHS    = 200

def gradient_penalty(D, real, fake, device):
    alpha  = torch.rand(real.size(0), 1, device=device)
    interp = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    grads  = torch.autograd.grad(
        D(interp), interp,
        grad_outputs=torch.ones(real.size(0), 1, device=device),
        create_graph=True, retain_graph=True,
    )[0]
    return ((grads.view(grads.size(0), -1).norm(2, dim=1) - 1) ** 2).mean()

def neg_critic_loss(D, G, loader, device):
    """D(real).mean() - D(fake).mean() sur un loader (sans GP)."""
    D.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for x, _ in loader:
            x    = x.view(-1, 784).to(device)
            z    = torch.randn(x.size(0), 100, device=device)
            total += (D(x).mean() - D(G(z)).mean()).item()
            n += 1
    D.train()
    return total / max(n, 1)

# --- Boucle d'entraînement ---
train_curve, val_curve, gen_iters = [], [], []
critic_step = gen_step = 0

for epoch in range(1, EPOCHS + 1):
    for x, _ in train_loader:
        x  = x.view(-1, 784).to(device)
        bs = x.size(0)

        # Critic step
        D_opt.zero_grad()
        z    = torch.randn(bs, 100, device=device)
        with torch.no_grad():
            fake = G5(z)
        gp     = gradient_penalty(D5, x, fake, device)
        d_loss = -D5(x).mean() + D5(fake).mean() + LAMBDA_GP * gp
        d_loss.backward()
        D_opt.step()
        critic_step += 1

        # Generator step every N_CRITIC
        if critic_step % N_CRITIC == 0:
            G_opt.zero_grad()
            z2 = torch.randn(bs, 100, device=device)
            (-D5(G5(z2)).mean()).backward()
            G_opt.step()
            gen_step += 1

    # Évaluation à chaque epoch
    train_curve.append(neg_critic_loss(D5, G5, train_loader, device))
    val_curve.append(neg_critic_loss(D5, G5, val_loader,   device))
    gen_iters.append(gen_step)

    if epoch % 50 == 0:
        print(f"Epoch {epoch}/{EPOCHS} | train={train_curve[-1]:.3f} | val={val_curve[-1]:.3f}")

# --- Plot ---
plt.figure(figsize=(7, 4))
plt.plot(gen_iters, train_curve, label='train')
plt.plot(gen_iters, val_curve,   label='validation', linestyle='--')
plt.xlabel('Generator iterations')
plt.ylabel('Negative critic loss')
plt.title('Negative critic loss — WGAN-GP, 1000 images MNIST\n(reproduction Fig. 5b, Gulrajani et al. 2017)')
plt.legend()
plt.tight_layout()
plt.show()


## Cellule 6 — Reproduction Figure 5b (Gulrajani et al. 2017)

Entraîne un WGAN-GP **from scratch** sur seulement **1000 images MNIST** (comme dans le papier) pour forcer l'overfitting et visualiser la divergence train/val de la *negative critic loss* (= estimation Wasserstein D(real) − D(fake)).

> **N'affecte pas les checkpoints** — modèles et optimiseurs complètement séparés.